> **Author:** Brendan OConnell  
> **Year:** 2026  
> **Purpose:**   
> **Key decisions:**  
> *   
> * 

In [96]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import xgboost as xgb
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
)

In [97]:
# custom functions


def sentinel_divide(n, d):
    sentinel = -1
    safe_d = np.where(d == 0, sentinel, d)
    return n / safe_d

### Using the Preprocessed Minimal NFI dataset
This includes all 89 original elements because XGBoost can handle the sparsity.

In [98]:
df = pd.read_parquet("../../data/processed/preprocessed_minimal.parquet")
df.shape

(2294985, 94)

In [99]:
# NFI column sort
nfi_meta = ["stub_id", "particle_id", "class", "label", "target"]
element_cols = [c for c in df.columns if c not in nfi_meta]
gsr = ["pb", "sb", "ba"]
non_gsr = sorted([c for c in element_cols if c not in gsr])
nfi_df = df[nfi_meta + gsr + non_gsr]

print(f"# of elements: {len(element_cols)}")

# of elements: 89


In [100]:
nfi_df.columns

Index(['stub_id', 'particle_id', 'class', 'label', 'target', 'pb', 'sb', 'ba',
       'ac', 'ag', 'al', 'ar', 'as', 'at', 'au', 'b', 'bi', 'br', 'ca', 'cd',
       'ce', 'cl', 'co', 'cr', 'cs', 'cu', 'dy', 'er', 'eu', 'f', 'fe', 'fr',
       'ga', 'gd', 'ge', 'hf', 'hg', 'ho', 'i', 'in', 'ir', 'k', 'kr', 'la',
       'lu', 'mg', 'mn', 'mo', 'n', 'na', 'nb', 'nd', 'ne', 'ni', 'np', 'o',
       'os', 'p', 'pa', 'pd', 'pm', 'po', 'pr', 'pt', 'pu', 'ra', 'rb', 're',
       'rh', 'rn', 'ru', 's', 'sc', 'se', 'si', 'sm', 'sn', 'sr', 'ta', 'tb',
       'tc', 'te', 'th', 'ti', 'tl', 'tm', 'u', 'v', 'w', 'xe', 'y', 'yb',
       'zn', 'zr'],
      dtype='str')

### Re-engineer Features

In [101]:
eng_df = nfi_df.copy()

In [102]:
# Pb * Sb
eng_df["pb_times_sb"] = eng_df["pb"] * (eng_df["sb"])

In [103]:
# Log (Pb + Sb)
eng_df["log_pb_plus_sb"] = np.log1p(eng_df["pb"] + eng_df["sb"])

In [104]:
# GSR ratios over total mass
total_mass = eng_df[element_cols].sum(axis=1)
total_mass_no_sb = total_mass - eng_df["sb"]
total_mass_no_ba = total_mass - eng_df["ba"]
total_mass_no_pb = total_mass - eng_df["pb"]

eng_df["pb_ba_over_non_sb_mass"] = (eng_df["pb"] + eng_df["ba"]) / total_mass_no_sb
eng_df["pb_sb_over_non_ba_mass"] = (eng_df["pb"] + eng_df["sb"]) / total_mass_no_ba
eng_df["ba_sb_over_non_pb_mass"] = (eng_df["ba"] + eng_df["sb"]) / total_mass_no_pb

In [105]:
# Brass particles
eng_df["cu_zn_over_mass"] = (eng_df["cu"] + eng_df["zn"]) / total_mass

# Titanium Zinc
eng_df["ti_zn_over_mass"] = (eng_df["ti"] + eng_df["zn"]) / total_mass

In [106]:
# Non-Barium GSR over Non-Barium Confounders
gsr = eng_df["pb"] + eng_df["sb"]
confounders = (
    eng_df["ca"] + eng_df["si"] + eng_df["al"] + eng_df["fe"]
)  # + eng_df['ti'] + eng_df['zn'] + eng_df['cu']
eng_df["gsr_over_confounders"] = sentinel_divide(gsr.values, confounders.values)

# check for any 'NaN' or 'inf'
any(np.isinf(eng_df["gsr_over_confounders"]) | eng_df["gsr_over_confounders"].isna())

False

In [107]:
eng_df.columns

Index(['stub_id', 'particle_id', 'class', 'label', 'target', 'pb', 'sb', 'ba',
       'ac', 'ag',
       ...
       'zn', 'zr', 'pb_times_sb', 'log_pb_plus_sb', 'pb_ba_over_non_sb_mass',
       'pb_sb_over_non_ba_mass', 'ba_sb_over_non_pb_mass', 'cu_zn_over_mass',
       'ti_zn_over_mass', 'gsr_over_confounders'],
      dtype='str', length=102)

In [108]:
eng_cols = [c for c in eng_df.columns if c not in nfi_meta + element_cols]
print(f"# of engineered features: {len(eng_cols)}")
print(f"\nEngineered features:")
for feat in eng_cols:
    print(f"\t{feat}")

# of engineered features: 8

Engineered features:
	pb_times_sb
	log_pb_plus_sb
	pb_ba_over_non_sb_mass
	pb_sb_over_non_ba_mass
	ba_sb_over_non_pb_mass
	cu_zn_over_mass
	ti_zn_over_mass
	gsr_over_confounders


__8 engineered features + 89 raw element features__

In [109]:
feature_cols = element_cols + eng_cols
print(f"# of feature columns: {len(feature_cols)}")

# of feature columns: 97


### Group aware train/test/val split 

Two-stage GroupShuffleSplit:
1. Split off 20% test (by stub_id)
2. From the remaining 80%, split off 25% as val (= 20% of total)

Result: ~60% train, ~20% val, ~20% test. No stub overlap between any pair.

In [110]:
X = eng_df[feature_cols].values.astype(np.float32)
y = eng_df["target"].values.astype(np.float32)
groups = eng_df["stub_id"].values

gss1 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
trainval_idx, test_idx = next(gss1.split(X, y, groups))

X_trainval, X_test = X[trainval_idx], X[test_idx]
y_trainval, y_test = y[trainval_idx], y[test_idx]
groups_trainval = groups[trainval_idx]

gss2 = GroupShuffleSplit(
    n_splits=1, test_size=0.25, random_state=42
)  # 25% of 80% = 20% of total
train_idx_rel, val_idx_rel = next(gss2.split(X_trainval, y_trainval, groups_trainval))

X_train = X_trainval[train_idx_rel]
X_val = X_trainval[val_idx_rel]
y_train = y_trainval[train_idx_rel]
y_val = y_trainval[val_idx_rel]

groups_train = set(groups_trainval[train_idx_rel])
groups_val = set(groups_trainval[val_idx_rel])
groups_test = set(groups[test_idx])

In [111]:
# Verify no stub leakage
assert len(groups_train & groups_val) == 0, "Train-Val stub leak!"
assert len(groups_train & groups_test) == 0, "Train-Test stub leak!"
assert len(groups_val & groups_test) == 0, "Val-Test stub leak!"

print("No stub-level leakage")

No stub-level leakage


Summarize the train/val/test split

In [112]:
total_obs = len(y_train) + len(y_val) + len(y_test)

summary = pd.DataFrame(
    {
        "Split": ["Train", "Val", "Test"],
        "Observations": [len(y_train), len(y_val), len(y_test)],
        "% of Total": [
            len(y_train) / total_obs * 100,
            len(y_val) / total_obs * 100,
            len(y_test) / total_obs * 100,
        ],
        "GSR (target=1)": [int(y_train.sum()), int(y_val.sum()), int(y_test.sum())],
        "GSR %": [y_train.mean() * 100, y_val.mean() * 100, y_test.mean() * 100],
        "Unique Stubs": [len(groups_train), len(groups_val), len(groups_test)],
    }
)

summary["% of Total"] = summary["% of Total"].map("{:.1f}%".format)
summary["GSR %"] = summary["GSR %"].map("{:.2f}%".format)

print(f"Total observations : {total_obs:,}")
print(f"Total features     : {X_train.shape[1]}")
print(f"Total unique stubs : {len(groups_train | groups_val | groups_test):,}\n")
print(summary.to_string(index=False))

Total observations : 2,294,985
Total features     : 97
Total unique stubs : 3,786

Split  Observations % of Total  GSR (target=1)  GSR %  Unique Stubs
Train       1444147      62.9%          717835 49.71%          2271
  Val        407614      17.8%          151162 37.08%           757
 Test        443224      19.3%          209949 47.37%           758


# Final Baseline

Default threshold = 0.5

In [113]:
baseline = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.1,
    min_child_weight=5,
    reg_alpha=0.1,
    reg_lambda=1.0,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    eval_metric="aucpr",
    early_stopping_rounds=30,
    random_state=42,
    n_jobs=-1,
)

baseline.fit(
    X_train, y_train, eval_set=[(X_train, y_train), (X_val, y_val)], verbose=10
)

[0]	validation_0-aucpr:0.99971	validation_1-aucpr:0.99985
[10]	validation_0-aucpr:0.99980	validation_1-aucpr:0.99988
[20]	validation_0-aucpr:0.99989	validation_1-aucpr:0.99994
[30]	validation_0-aucpr:0.99997	validation_1-aucpr:0.99999
[40]	validation_0-aucpr:0.99999	validation_1-aucpr:0.99999
[50]	validation_0-aucpr:0.99999	validation_1-aucpr:1.00000
[60]	validation_0-aucpr:1.00000	validation_1-aucpr:1.00000
[70]	validation_0-aucpr:1.00000	validation_1-aucpr:1.00000
[80]	validation_0-aucpr:1.00000	validation_1-aucpr:1.00000
[90]	validation_0-aucpr:1.00000	validation_1-aucpr:1.00000
[100]	validation_0-aucpr:1.00000	validation_1-aucpr:1.00000
[110]	validation_0-aucpr:1.00000	validation_1-aucpr:1.00000
[120]	validation_0-aucpr:1.00000	validation_1-aucpr:1.00000
[130]	validation_0-aucpr:1.00000	validation_1-aucpr:1.00000
[140]	validation_0-aucpr:1.00000	validation_1-aucpr:1.00000
[150]	validation_0-aucpr:1.00000	validation_1-aucpr:1.00000
[160]	validation_0-aucpr:1.00000	validation_1-aucpr

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",30
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes fro

__Eval Baseline train & val learning curves__

In [114]:
results = baseline.evals_result()
train_aucpr = results["validation_0"]["aucpr"]
val_aucpr = results["validation_1"]["aucpr"]

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.plot(train_aucpr, label="Train AUCPR")
ax.plot(val_aucpr, label="Val AUCPR")
ax.set_xlabel("Boosting Round")
ax.set_ylabel("AUCPR")
ax.set_title("XGBoost: Learning Curves Diagnostic")
ax.legend()
plt.tight_layout()
plt.savefig("outputs/xgb_optimize__xgb_learning_curves_final.png", dpi=150)
plt.show()

In [116]:
# Feature Importance
importances = baseline.feature_importances_
feat_imp = pd.Series(importances, index=feature_cols).sort_values(ascending=False)
print(f"Final Baseline: Top 20 Important Features\n\n{feat_imp.head(20)}")

Final Baseline: Top 20 Important Features

log_pb_plus_sb            0.840285
pb_sb_over_non_ba_mass    0.110785
pb_ba_over_non_sb_mass    0.011780
pb                        0.008506
hg                        0.005362
mo                        0.004609
ba_sb_over_non_pb_mass    0.003076
gd                        0.002678
sb                        0.002519
ba                        0.001738
pb_times_sb               0.001509
zn                        0.000700
as                        0.000654
ti_zn_over_mass           0.000530
gsr_over_confounders      0.000523
al                        0.000438
cu_zn_over_mass           0.000437
cu                        0.000429
cr                        0.000395
sn                        0.000365
dtype: float32


### Evaluate against test

In [117]:
y_prob = baseline.predict_proba(X_test)[:, 1]
y_pred = baseline.predict(X_test)

y_prob_test = baseline.predict_proba(X_test)[:, 1]
y_pred_test = baseline.predict(X_test)

precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob_test)

# FP and FN breakdown
fp_mask = (y_pred_test == 1) & (y_test == 0)
fn_mask = (y_pred_test == 0) & (y_test == 1)

df_test = eng_df.iloc[test_idx]
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
print(f"TN={tn:,}  FP={fp:,}  FN={fn:,}  TP={tp:,}")
print(f"FPR: {fp / (fp + tn):.6f}")

print("\nFP by class:")
print(df_test["class"].loc[fp_mask].value_counts())
print("\nFN by class:")
print(df_test["class"].loc[fn_mask].value_counts())

# FP probability distribution
fp_probs = y_prob_test[fp_mask]
print(
    f"\nFP probs: min={fp_probs.min():.4f}, "
    f"median={np.median(fp_probs):.4f}, max={fp_probs.max():.4f}"
)

# Profile the FPs vs correctly classified particles of the same class
fp_baal = df_test.loc[fp_mask & (df_test["class"] == "BaAl")]
tn_baal = df_test.loc[(y_pred_test == 0) & (y_test == 0) & (df_test["class"] == "BaAl")]

print("\nBaAl FPs vs correctly classified BaAl:")
for el in ["pb", "sb", "ba", "al", "cu", "fe"]:
    print(f"  {el}: FP mean={fp_baal[el].mean():.2f}, TN mean={tn_baal[el].mean():.2f}")

TN=233,122  FP=153  FN=34  TP=209,915
FPR: 0.000656

FP by class:
class
BaAl      107
BaCaSi     40
CuZn        3
TiZnGd      3
Name: count, dtype: int64

FN by class:
class
PbBa    16
BaSb    15
PbSb     3
Name: count, dtype: int64

FP probs: min=0.5029, median=0.7204, max=0.9997

BaAl FPs vs correctly classified BaAl:
  pb: FP mean=1.15, TN mean=0.00
  sb: FP mean=0.65, TN mean=0.01
  ba: FP mean=32.42, TN mean=31.42
  al: FP mean=13.83, TN mean=4.67
  cu: FP mean=3.93, TN mean=0.80
  fe: FP mean=3.44, TN mean=6.66


__Cross Validation__

In [118]:
cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)

cv_results = []
for fold, (tr_idx, va_idx) in enumerate(
    cv.split(X_trainval, y_trainval, groups_trainval)
):
    model_cv = xgb.XGBClassifier(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.1,
        min_child_weight=5,
        reg_alpha=0.1,
        reg_lambda=1.0,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        eval_metric="aucpr",
        early_stopping_rounds=30,
        random_state=42,
        n_jobs=-1,
    )
    model_cv.fit(
        X_trainval[tr_idx],
        y_trainval[tr_idx],
        eval_set=[(X_trainval[va_idx], y_trainval[va_idx])],
        verbose=False,
    )
    y_prob_cv = model_cv.predict_proba(X_trainval[va_idx])[:, 1]
    y_pred_cv = model_cv.predict(X_trainval[va_idx])

    pr_auc = average_precision_score(y_trainval[va_idx], y_prob_cv)
    roc_auc = roc_auc_score(y_trainval[va_idx], y_prob_cv)
    cm = confusion_matrix(y_trainval[va_idx], y_pred_cv)
    tn, fp, fn, tp = cm.ravel()
    fpr = fp / (fp + tn)
    acc = (tp + tn) / (tp + tn + fp + fn)

    cv_results.append(
        {
            "fold": fold + 1,
            "PR-AUC": pr_auc,
            "ROC-AUC": roc_auc,
            "Accuracy": acc,
            "FP": fp,
            "FN": fn,
            "FPR": fpr,
        }
    )
    print(
        f"Fold {fold + 1}: PR-AUC={pr_auc:.6f}, ROC-AUC={roc_auc:.6f}, "
        f"FP={fp}, FN={fn}, FPR={fpr:.6f}"
    )

Fold 1: PR-AUC=0.999999, ROC-AUC=0.999999, FP=88, FN=112, FPR=0.000269
Fold 2: PR-AUC=0.999998, ROC-AUC=0.999998, FP=189, FN=108, FPR=0.000577
Fold 3: PR-AUC=0.999998, ROC-AUC=0.999998, FP=223, FN=86, FPR=0.000681


In [119]:
# Cross-Validation results
cv_df = pd.DataFrame(cv_results)
print("XGBoost final baseline cross validation results:\n")
print(f"Mean PR-AUC: {cv_df['PR-AUC'].mean():.6f} +/- {cv_df['PR-AUC'].std():.6f}")
print(f"Mean ROC-AUC: {cv_df['ROC-AUC'].mean():.6f} +/- {cv_df['ROC-AUC'].std():.6f}")
print(f"Mean FPR: {cv_df['FPR'].mean():.6f} +/- {cv_df['FPR'].std():.6f}")

XGBoost final baseline cross validation results:

Mean PR-AUC: 0.999998 +/- 0.000000
Mean ROC-AUC: 0.999998 +/- 0.000000
Mean FPR: 0.000509 +/- 0.000214


### Summary of Metrics

In [120]:
# === FULL TEST SET METRICS TABLE ===
# Compute test metrics
y_prob_test = baseline.predict_proba(X_test)[:, 1]
y_pred_test = baseline.predict(X_test)

cm = confusion_matrix(y_test, y_pred_test)
tn, fp, fn, tp = cm.ravel()

metrics = {
    "Features": X_test.shape[1],
    "Accuracy": (tp + tn) / (tp + tn + fp + fn),
    "Precision (GSR)": precision_score(y_test, y_pred_test),
    "Recall (GSR)": recall_score(y_test, y_pred_test),
    "F1 (GSR)": f1_score(y_test, y_pred_test),
    "ROC-AUC": roc_auc_score(y_test, y_prob_test),
    "PR-AUC": average_precision_score(y_test, y_prob_test),
    "False Positives": fp,
    "FPR": fp / (fp + tn),
    "Early Stopping Round": baseline.best_iteration,
    "CV Mean PR-AUC": "0.999998 ± 0.000001",
    "CV Mean FPR": "0.000525 ± 0.000216",
}

metrics_df = pd.DataFrame({"Final Baseline": metrics})
print(metrics_df.to_string())

                           Final Baseline
Features                               97
Accuracy                         0.999578
Precision (GSR)                  0.999272
Recall (GSR)                     0.999838
F1 (GSR)                         0.999555
ROC-AUC                          0.999998
PR-AUC                           0.999998
False Positives                       153
FPR                              0.000656
Early Stopping Round                  174
CV Mean PR-AUC        0.999998 ± 0.000001
CV Mean FPR           0.000525 ± 0.000216


In [ ]:
# === CONFUSION MATRIX HEATMAP ===
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm.reshape(2, 2),
    annot=True,
    fmt=",",
    cmap="Blues",
    xticklabels=["Non-GSR", "GSR"],
    yticklabels=["Non-GSR", "GSR"],
    ax=ax,
    annot_kws={"size": 14},
)
ax.set_xlabel("Predicted", fontsize=12)
ax.set_ylabel("Actual", fontsize=12)
ax.set_title("Confusion Matrix (Final Baseline, Test Set)", fontsize=13)
plt.tight_layout()
plt.savefig("outputs/xgb_optimize__confusion_matrix_final.png", dpi=150)
plt.show()

In [ ]:
importances = baseline.feature_importances_
feat_imp = pd.Series(importances, index=feature_cols).sort_values(ascending=True)

# Show top 10
top_n = feat_imp.tail(10)

fig, ax = plt.subplots(figsize=(5, 3))
top_n.plot(kind="barh", ax=ax, color="#2c3e50", edgecolor="black")
ax.set_xlabel("Feature Importance (Gain)")
ax.set_title("Top 10 Feature Importances")
plt.tight_layout()
plt.savefig("outputs/xgb_optimize__feature_importance.png", dpi=150)
plt.show()

In [ ]:
# === FP PROBABILITY DISTRIBUTION HISTOGRAM ===
fp_probs = y_prob_test[fp_mask]

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(fp_probs, bins=30, color="green", edgecolor="black", alpha=0.8)
ax.axvline(x=0.5, color="black", linestyle="--", label="Threshold (0.5)")
ax.set_xlabel("P(GSR)")
ax.set_ylabel("Count")
ax.set_title(f"False Positive Probability Distribution (n={len(fp_probs)})")
ax.legend()
plt.tight_layout()
plt.savefig("outputs/xgb_optimize__fp_prob_dist.png", dpi=150)
plt.show()